# Model Optimization Workshop: Introduction and Setup

Welcome to the Model Optimization Workshop! This notebook will guide you through the process of setting up your environment and downloading models for optimization experiments. We'll cover the following topics:

1. Introduction to model optimization
2. Environment setup and package installation
3. AWS configuration
4. Model selection and download

## Workshop Structure

1. **Introduction and Setup** (this notebook)
2. **Baseline Evaluation**: Measuring initial performance metrics
3. **Quantization**: Reducing precision to improve efficiency
4. **Pruning**: Removing unnecessary weights
5. **Knowledge Distillation**: Creating smaller student models
6. **Model Hosting**: Deploying optimized models
7. **Inference Performance**: Comparing optimized vs. baseline models
8. **Cost Analysis**: Calculating ROI for optimization techniques
9. **Resource Cleanup**: Removing deployed resources

## Prerequisites

- Basic understanding of machine learning concepts
- Python programming experience
- AWS account with appropriate permissions
- Familiarity with Jupyter notebooks

# Part 1: Introduction to Model Optimization

## Workshop Goals

By the end of this workshop, you will be able to:

1. Download and prepare models from Hugging Face
2. Apply various optimization techniques to reduce model size and improve inference speed
3. Quantify the impact of optimization on model performance and cost
4. Deploy optimized models to production environments
5. Calculate the ROI of model optimization efforts

## Why Model Optimization Matters

Modern deep learning models are becoming increasingly large and computationally expensive. For example:

- BERT-large: 340M parameters
- GPT-2: 1.5B parameters
- GPT-3: 175B parameters

This growth presents several challenges:

- **Cost**: Larger models require more compute resources for training and inference
- **Latency**: Slower response times impact user experience
- **Deployment constraints**: Many edge devices cannot run large models
- **Energy consumption**: Larger models have a greater environmental impact

Model optimization techniques help address these challenges by reducing model size and computational requirements while preserving accuracy.

# Part 2: Environment Setup

## 1. Install Required Packages

We've provided a `requirements.txt` file with all the necessary packages. Let's install them:

In [ ]:
!pip install -q -r requirements.txt

## 2. Verify Installations

In [ ]:
import torch
import transformers
import boto3
import sagemaker
import optimum
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"Boto3 version: {boto3.__version__}")
print(f"SageMaker SDK version: {sagemaker.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Check if GPU is available
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected. Using CPU only.")
    
# Try to get optimum version using pkg_resources
try:
    import pkg_resources
    optimum_version = pkg_resources.get_distribution("optimum").version
    print(f"Optimum version: {optimum_version}")
except Exception:
    print("Could not determine optimum version")

## 3. Configure AWS Region and Get Execution Role

When running in SageMaker, we'll use the execution role assigned to the notebook instance. This role has the necessary permissions to interact with AWS services.

In [ ]:
import sagemaker

# Get the SageMaker session
sagemaker_session = sagemaker.Session()

# Get the execution role
try:
    role = sagemaker.get_execution_role()
except ValueError:
    print("Not running in SageMaker, using default role")
    role = "arn:aws:iam::role/service-role/AmazonSageMaker-ExecutionRole"

# Get the AWS region
region = boto3.session.Session().region_name

print(f"SageMaker execution role: {role}")
print(f"AWS region: {region}")

Let's verify that our AWS credentials are working:

In [ ]:
# Verify AWS credentials
try:
    client = boto3.client('sts')
    response = client.get_caller_identity()
    print(f"AWS credentials verified. Account ID: {response['Account']}")
    print(f"Role ARN: {response['Arn']}")
except Exception as e:
    print(f"Error verifying AWS credentials: {e}")

## 5. Create S3 Bucket for Workshop

We'll create an S3 bucket to store our models and data. We'll use a unique name based on the account ID.

In [ ]:
import boto3
import uuid

# Generate a unique bucket name
account_id = boto3.client('sts').get_caller_identity().get('Account')
bucket_name = f"model-optimization-workshop-{account_id}-{uuid.uuid4().hex[:8]}"

# Create the bucket
s3_client = boto3.client('s3')
region = boto3.session.Session().region_name

try:
    if region == 'us-east-1':
        s3_client.create_bucket(Bucket=bucket_name)
    else:
        s3_client.create_bucket(
            Bucket=bucket_name,
            CreateBucketConfiguration={'LocationConstraint': region}
        )
    print(f"Created S3 bucket: {bucket_name}")
except Exception as e:
    print(f"Error creating S3 bucket: {e}")

# Save the bucket name for later use
import os
os.environ['WORKSHOP_BUCKET'] = bucket_name

## 6. Create Workshop Configuration

In [ ]:
# Create workshop configuration file
with open('workshop_config.py', 'w') as f:
    f.write(f"S3_BUCKET = '{bucket_name}'\n")
    f.write(f"AWS_REGION = '{region}'\n")
    f.write(f"SAGEMAKER_ROLE_ARN = '{role}'\n")

## 7. Verify Workshop Configuration

In [ ]:
# Import workshop configuration
from workshop_config import S3_BUCKET, AWS_REGION, SAGEMAKER_ROLE_ARN

print(f"S3 Bucket: {S3_BUCKET}")
print(f"AWS Region: {AWS_REGION}")
print(f"SageMaker Role ARN: {SAGEMAKER_ROLE_ARN}")

# Verify utils.py exists
import os
if os.path.exists('utils.py'):
    print("utils.py exists")
    import utils
    print(f"Available utility functions: {[f for f in dir(utils) if not f.startswith('_')]}")
else:
    print("utils.py does not exist - it should have been created in the repository")

# Part 3: Model Selection and Download

## 1. Import Dependencies for Model Selection

In [ ]:
import os
import torch
import json
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForTokenClassification
from transformers import AutoModelForQuestionAnswering, AutoModelForMaskedLM
import boto3
import time
from pathlib import Path

# Import workshop configuration
from workshop_config import S3_BUCKET, AWS_REGION

## 2. Define Models to Download

We'll download several models for different tasks to demonstrate optimization techniques across various model architectures and sizes.

In [ ]:
# Define models to download
models_to_download = {
    "sentiment_analysis": {
        "model_name": "distilbert-base-uncased-finetuned-sst-2-english",
        "task": "sequence-classification",
        "description": "DistilBERT model fine-tuned for sentiment analysis"
    },
    "ner": {
        "model_name": "dbmdz/bert-large-cased-finetuned-conll03-english",
        "task": "token-classification",
        "description": "BERT model fine-tuned for named entity recognition"
    },
    "question_answering": {
        "model_name": "distilbert-base-cased-distilled-squad",
        "task": "question-answering",
        "description": "DistilBERT model fine-tuned for question answering"
    },
    "masked_lm": {
        "model_name": "bert-base-uncased",
        "task": "masked-lm",
        "description": "BERT model for masked language modeling"
    }
}

## 3. Create Function to Download Models and Save to S3

In [ ]:
def download_and_save_to_s3(model_info, bucket_name, temp_dir="temp_models"):
    """Download model from Hugging Face and save directly to S3."""
    model_name = model_info["model_name"]
    task = model_info["task"]
    
    # Create temporary directory if it doesn't exist
    model_dir = os.path.join(temp_dir, model_name.replace("/", "_"))
    os.makedirs(model_dir, exist_ok=True)
    
    print(f"Downloading {model_name} for {task}...")
    
    # Download tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.save_pretrained(model_dir)
    
    # Download model based on task
    if task == "sequence-classification":
        model = AutoModelForSequenceClassification.from_pretrained(model_name)
    elif task == "token-classification":
        model = AutoModelForTokenClassification.from_pretrained(model_name)
    elif task == "question-answering":
        model = AutoModelForQuestionAnswering.from_pretrained(model_name)
    elif task == "masked-lm":
        model = AutoModelForMaskedLM.from_pretrained(model_name)
    else:
        raise ValueError(f"Unsupported task: {task}")
    
    # Save model to temporary directory
    model.save_pretrained(model_dir)
    
    # Get model size
    model_size_mb = sum(p.numel() * p.element_size() for p in model.parameters()) / (1024 * 1024)
    
    # Upload to S3
    s3_client = boto3.client('s3')
    s3_prefix = f"models/{model_name.replace('/', '_')}/"
    
    print(f"Uploading {model_name} to S3...")
    for root, _, files in os.walk(model_dir):
        for file in files:
            local_path = os.path.join(root, file)
            relative_path = os.path.relpath(local_path, model_dir)
            s3_key = s3_prefix + relative_path
            s3_client.upload_file(local_path, bucket_name, s3_key)
    
    # Create S3 URI for the model
    s3_uri = f"s3://{bucket_name}/{s3_prefix}"
    
    print(f"Uploaded {model_name} ({model_size_mb:.2f} MB) to {s3_uri}")
    
    return {
        "model_name": model_name,
        "task": task,
        "s3_uri": s3_uri,
        "local_path": model_dir,  # Keep local path for reference
        "size_mb": model_size_mb
    }

## 4. Download Models and Save to S3

In [ ]:
# Create temporary directory
temp_dir = "temp_models"
os.makedirs(temp_dir, exist_ok=True)

# Download models and save to S3
downloaded_models = {}
for key, model_info in models_to_download.items():
    downloaded_models[key] = download_and_save_to_s3(model_info, S3_BUCKET, temp_dir)
    
    # Add a small delay between downloads to avoid rate limiting
    time.sleep(2)

## 5. Create Function to Load Models from S3

In [ ]:
def load_model_from_s3(model_info, temp_dir="temp_models"):
    """Load a model from S3 to local storage for use in the notebook."""
    model_name = model_info["model_name"]
    s3_uri = model_info["s3_uri"]
    local_path = model_info["local_path"]
    
    # Check if model is already downloaded
    if os.path.exists(local_path) and os.path.isfile(os.path.join(local_path, "pytorch_model.bin")):
        print(f"Model {model_name} already exists locally at {local_path}")
        return local_path
    
    print(f"Loading model {model_name} from {s3_uri}...")
    
    # Parse S3 URI
    s3_parts = s3_uri.replace("s3://", "").split("/")
    bucket = s3_parts[0]
    prefix = "/".join(s3_parts[1:])
    
    # Create S3 client
    s3_client = boto3.client('s3')
    
    # List objects in the prefix
    response = s3_client.list_objects_v2(Bucket=bucket, Prefix=prefix)
    
    # Download each file
    os.makedirs(local_path, exist_ok=True)
    for obj in response.get('Contents', []):
        key = obj['Key']
        filename = os.path.basename(key)
        if filename:  # Skip directory entries
            local_file = os.path.join(local_path, filename)
            s3_client.download_file(bucket, key, local_file)
    
    print(f"Model loaded to {local_path}")
    return local_path

## 6. Test Loading a Model from S3

In [ ]:
# Test loading a model from S3
model_key = "sentiment_analysis"  # Choose one model to test
model_info = downloaded_models[model_key]

local_path = load_model_from_s3(model_info)
print(f"Model loaded to {local_path}")

# Test loading the model with transformers
tokenizer = AutoTokenizer.from_pretrained(local_path)
task = model_info["task"]

if task == "sequence-classification":
    model = AutoModelForSequenceClassification.from_pretrained(local_path)
elif task == "token-classification":
    model = AutoModelForTokenClassification.from_pretrained(local_path)
elif task == "question-answering":
    model = AutoModelForQuestionAnswering.from_pretrained(local_path)
elif task == "masked-lm":
    model = AutoModelForMaskedLM.from_pretrained(local_path)

print(f"Successfully loaded model and tokenizer from {local_path}")

## 7. Save Model Information

In [ ]:
# Save model information to file
with open('model_info.json', 'w') as f:
    json.dump(downloaded_models, f, indent=2)

print("Model information saved to model_info.json")

## 8. Display Model Summary

In [ ]:
# Create a DataFrame with model information
model_data = []
for model_key, model_info in downloaded_models.items():
    model_data.append({
        "Task": model_key,
        "Model": model_info["model_name"],
        "Size (MB)": f"{model_info['size_mb']:.2f}",
        "S3 Location": model_info["s3_uri"]
    })

# Display as a table
pd.DataFrame(model_data)

## 9. Clean Up Temporary Files (Optional)

In [ ]:
# Uncomment to remove temporary files
# import shutil
# shutil.rmtree(temp_dir)
# print(f"Removed temporary directory: {temp_dir}")

# Next Steps

Now that we've set up our environment and downloaded our models, we're ready to proceed to the next notebook where we'll establish baseline performance metrics for each model.

In the next notebook, we will:
1. Load the models from S3
2. Measure inference time, memory usage, and model size
3. Establish baseline performance metrics for comparison with optimized models

Continue to the next notebook: `02_baseline_evaluation.ipynb`